# Membuat Dataset dari 240 PDF Artikel

**Catatan**
- Teks disimpan ke **CSV** (tidak ada batas panjang sel) dan ke **Excel** dengan teks yang dipecah ke beberapa kolom.
- Ligatur dipulihkan dengan normalisasi Unicode **NFKC**, misalnya karakter tunggal `ﬁ` (U+FB01) menjadi dua huruf `fi`.

**Alur kerja:**
1. Instalasi & impor pustaka
2. Pengaturan lokasi folder
3. Fungsi ekstraksi teks per halaman
4. Fungsi pembersihan teks
5. Uji coba pada 1 file
6. Ekstraksi semua PDF → DataFrame
7. Pemeriksaan kualitas data
8. Menyimpan dataset (CSV, Excel, Pickle)
9. Verifikasi file hasil

> Dataset dibuat dengan **pypdf 5.9.0**.

## 1. Instalasi dan impor pustaka

Pustaka yang dipakai:
| Pustaka | Fungsi |
|---|---|
| `pypdf` | membaca PDF dan mengambil teks per halaman (murni Python, mudah dipasang di Windows) |
| `pandas` | menyusun dan menyimpan dataset |
| `openpyxl` | menulis file Excel (.xlsx) |

In [1]:
%pip install --quiet pypdf==5.9.0 pandas openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import time
import logging
import unicodedata
from pathlib import Path

import pandas as pd
from pypdf import PdfReader

logging.getLogger("pypdf").setLevel(logging.ERROR)

import pypdf
print("pypdf  :", pypdf.__version__)
print("pandas :", pd.__version__)

if pypdf.__version__ != "5.9.0":
    print(f"PERINGATAN: pypdf {pypdf.__version__} terpasang, bukan 5.9.0. Teks bisa sedikit berbeda dari dataset naskah.")
    print("Jalankan sel pemasangan paket di atas, lalu Restart kernel.")


pypdf  : 5.9.0
pandas : 3.0.6


## 2. Lokasi folder

In [3]:
FOLDER_PDF    = Path(r"C:\Users\marfu\Downloads\journal-20261013T092646Z-1-001\journal")
FOLDER_OUTPUT = Path(r"C:\Users\marfu\Documents\LDA_SBMI")
FOLDER_OUTPUT.mkdir(parents=True, exist_ok=True)

KATA_KUNCI = {
    "s": "Digital servitization",
    "e": "Ethical leadership",
    "d": "Dynamic capabilities",
}

daftar_pdf = sorted(FOLDER_PDF.glob("*.pdf"))
print("Jumlah PDF ditemukan:", len(daftar_pdf))
print("Contoh 5 file pertama:", [p.name for p in daftar_pdf[:5]])
assert len(daftar_pdf) == 240, "Seharusnya ada 240 PDF. Periksa FOLDER_PDF."

Jumlah PDF ditemukan: 240
Contoh 5 file pertama: ['file001d.pdf', 'file001e.pdf', 'file001s.pdf', 'file002d.pdf', 'file002e.pdf']


## 3. Fungsi ekstraksi teks per halaman

`pypdf` menyediakan dua mode ekstraksi:
- **Mode biasa** (`extract_text()`): paling akurat untuk sebagian besar artikel jurnal, termasuk tata letak dua kolom.
- **Mode layout** (`extract_text(extraction_mode="layout")`): mengikuti posisi huruf di halaman. Mode ini berguna ketika teks rata kiri-kanan (*justified*) membuat kata-kata tersambung tanpa spasi, misalnya `substantialeffortstoexpand`.

Strategi yang dipakai: gunakan **mode biasa**. Bila sebuah halaman memuat lebih dari 3 "kata" sangat panjang (≥ 22 huruf berturut-turut, tanda spasi hilang), coba **mode layout** untuk halaman itu dan pilih hasil yang kata panjangnya lebih sedikit.

In [4]:
POLA_KATA_TERSAMBUNG = re.compile(r"[A-Za-z]{22,}")   # ≥ 22 huruf tanpa spasi = kemungkinan kata tersambung

def teks_halaman(halaman):
    # Ambil teks satu halaman PDF; beralih ke mode layout bila spasi antarkata banyak yang hilang.
    teks = halaman.extract_text() or ""
    if len(POLA_KATA_TERSAMBUNG.findall(teks)) > 3:
        try:
            teks_layout = halaman.extract_text(extraction_mode="layout") or ""
            if len(POLA_KATA_TERSAMBUNG.findall(teks_layout)) < len(POLA_KATA_TERSAMBUNG.findall(teks)):
                teks = teks_layout
        except Exception:
            pass            # bila mode layout gagal, tetap pakai hasil mode biasa
    return teks


def ekstrak_pdf(path_pdf):
    # Gabungkan teks semua halaman satu PDF. Mengembalikan (teks_mentah, jumlah_halaman).
    reader = PdfReader(str(path_pdf))
    semua_halaman = [teks_halaman(h) for h in reader.pages]
    return "\n".join(semua_halaman), len(reader.pages)

## 4. Fungsi pembersihan teks

Urutan langkah di `bersihkan_teks()`:

| No | Langkah | Contoh |
|---|---|---|
| 1 | Normalisasi Unicode **NFKC**: memecah ligatur menjadi huruf biasa | `ﬁrms` → `firms`, `eﬀect` → `effect` |
| 2 | Menghapus *soft hyphen* dan karakter tak terlihat (*zero-width*) | `https​://` → `https://` |
| 2b | Menempelkan kembali tanda diakritik yang terlepas dari hurufnya | `Jyv ̈askyl ̈a` → `Jyväskylä` |
| 3 | Menyambung kata yang terpotong tanda hubung di akhir baris | `manu-\nfacturing` → `manufacturing` |
| 4 | Mengganti karakter kontrol (tak tercetak) dengan spasi | `\x02`, `\x0b` → spasi |
| 5 | Merapikan spasi, tab, dan baris baru menjadi satu spasi | `kata   \n  kata` → `kata kata` |

Langkah 4 juga penting agar teks bisa ditulis ke Excel, karena `openpyxl` menolak karakter kontrol.

In [5]:
POLA_TAK_TERLIHAT = re.compile("[\u00ad\u200b\u200c\u200d\u2060\ufeff]")
POLA_KONTROL      = re.compile(r"[\x00-\x08\x0b\x0e-\x1f\x7f-\x9f]")
POLA_DIAKRITIK    = re.compile("\\s?([\u0300-\u036f])(\\w)")      # tanda diakritik yang terlepas dari hurufnya

def bersihkan_teks(teks):
    teks = unicodedata.normalize("NFKC", teks)                 # 1. ligatur -> huruf biasa
    teks = POLA_TAK_TERLIHAT.sub("", teks)                    # 2. soft hyphen & zero-width
    teks = POLA_DIAKRITIK.sub(lambda m: unicodedata.normalize("NFC", m.group(2) + m.group(1)), teks)  # 2b. Jyv ̈a -> Jyvä
    teks = re.sub(r"(\w)-\n\s*(\w)", r"\1\2", teks)            # 3. sambung kata terpotong
    teks = POLA_KONTROL.sub(" ", teks)                        # 4. karakter kontrol
    teks = re.sub(r"\s+", " ", teks).strip()                  # 5. rapikan spasi
    return teks

# Uji fungsi pada contoh kecil
contoh = "The \ufb01rms\u2019 e\ufb00ect on manu-\nfacturing   was signi\ufb01cant.\u200b Jyv \u0308askyl \u0308a"
print("Sebelum :", repr(contoh))
print("Sesudah :", repr(bersihkan_teks(contoh)))

Sebelum : 'The ﬁrms’ eﬀect on manu-\nfacturing   was signiﬁcant.\u200b Jyv ̈askyl ̈a'
Sesudah : 'The firms’ effect on manufacturing was significant. Jyväskylä'


## 5. Uji coba pada satu file

Sebelum memproses 240 file, cek dulu satu file untuk memastikan hasilnya masuk akal. Perhatikan jumlah halaman, jumlah karakter, dan potongan awal teksnya.

In [6]:
uji = daftar_pdf[0]
mentah, n_hal = ekstrak_pdf(uji)
bersih = bersihkan_teks(mentah)

print("File            :", uji.name)
print("Jumlah halaman  :", n_hal)
print("Karakter mentah :", f"{len(mentah):,}")
print("Karakter bersih :", f"{len(bersih):,}")
print("Jumlah kata     :", f"{len(bersih.split()):,}")
print("\n500 karakter pertama:\n", bersih[:500])

File            : file001d.pdf
Jumlah halaman  : 22
Karakter mentah : 105,559
Karakter bersih : 105,187
Jumlah kata     : 15,358

500 karakter pertama:
 International Journal of Management Reviews, V ol. 00, 1–22 (2016) DOI: 10.1111/ijmr.12122 The Essence of Dynamic Capabilities and their Measurement Ola Laaksonen and Mirva Peltoniemi 1 Katajaharjuntie 20, 00200 Helsinki, Finland, and 1Jyväskylä School of Business and Economics, PO Box 35, FI-40014 University of Jyv äskylä, Jyv äskylä, Finland Corresponding author email: mirva.peltoniemi@jyu.fi The growing popularity of explaining firm performance through dynamic capabilities has motivated plent


## 6. Ekstraksi semua PDF menjadi DataFrame

Setiap PDF menghasilkan satu baris dengan kolom:

| Kolom | Isi |
|---|---|
| `articlename` | nama file PDF |
| `keyword` | kata kunci pencarian (dari akhiran nama file) |
| `pages` | jumlah halaman |
| `n_char` | jumlah karakter teks bersih |
| `n_words` | jumlah kata (dipisahkan spasi) |
| `textdata` | teks lengkap yang sudah dibersihkan |

In [7]:
baris = []
gagal = []
mulai = time.time()

for i, path_pdf in enumerate(daftar_pdf, start=1):
    try:
        mentah, n_hal = ekstrak_pdf(path_pdf)
        teks = bersihkan_teks(mentah)
        kode = path_pdf.stem[-1]                       # huruf terakhir nama file: s / e / d
        baris.append({
            "articlename": path_pdf.name,
            "keyword": KATA_KUNCI.get(kode, "Tidak diketahui"),
            "pages": n_hal,
            "n_char": len(teks),
            "n_words": len(teks.split()),
            "textdata": teks,
        })
    except Exception as e:
        gagal.append((path_pdf.name, str(e)))
    if i % 20 == 0 or i == len(daftar_pdf):
        print(f"{i:>3}/{len(daftar_pdf)} file selesai  ({time.time() - mulai:,.0f} detik)")

df = pd.DataFrame(baris)
print("\nBerhasil :", len(df), "file")
print("Gagal    :", len(gagal), gagal)

 20/240 file selesai  (45 detik)
 40/240 file selesai  (78 detik)
 60/240 file selesai  (110 detik)
 80/240 file selesai  (151 detik)
100/240 file selesai  (218 detik)
120/240 file selesai  (241 detik)
140/240 file selesai  (265 detik)
160/240 file selesai  (296 detik)
180/240 file selesai  (335 detik)
200/240 file selesai  (386 detik)
220/240 file selesai  (444 detik)
240/240 file selesai  (479 detik)

Berhasil : 240 file
Gagal    : 0 []


## 7. Pemeriksaan kualitas data

Empat pemeriksaan:
1. **Jumlah artikel per kata kunci.** Seharusnya 80 per kelompok.
2. **Statistik panjang teks.** Kolom `n_char`
3. **Teks kosong atau sangat pendek.** PDF hasil *scan* tidak memiliki lapisan teks dan perlu OCR.
4. **Sisa kerusakan ligatur.** Mencari kata rusak khas seperti *dierent* atau *benet*; hasil yang diharapkan 0.

In [8]:
print("1) Jumlah artikel per kata kunci")
print(df["keyword"].value_counts().to_string(), "\n")

print("2) Statistik panjang teks")
print(df[["pages", "n_char", "n_words"]].describe().round(0).to_string(), "\n")
print("   Teks lebih panjang dari batas sel Excel (32.767):", (df["n_char"] > 32767).sum(), "dari", len(df), "\n")

df["char_per_halaman"] = (df["n_char"] / df["pages"]).round(0)
print("3) PDF dengan < 500 karakter per halaman (kemungkinan hasil scan):")
tersangka_scan = df.loc[df["char_per_halaman"] < 500, ["articlename", "pages", "n_char"]]
print(tersangka_scan.to_string() if len(tersangka_scan) else "   tidak ada", "\n")

POLA_LIGATUR_RUSAK = re.compile(
    r"\b(dierent|benet|benets|signicant|specic|eect|eects|scientic|denition|"
    r"dened|eld|nancial|ecient|dicult|conict|reect|inuence|identied|modied)\b")
df["kata_rusak_ligatur"] = df["textdata"].str.lower().apply(lambda t: len(POLA_LIGATUR_RUSAK.findall(t)))
print("4) Total kata rusak ligatur:", df["kata_rusak_ligatur"].sum())
print("   File dengan ≥ 3 kata rusak:", (df["kata_rusak_ligatur"] >= 3).sum())

1) Jumlah artikel per kata kunci
keyword
Dynamic capabilities     80
Ethical leadership       80
Digital servitization    80 

2) Statistik panjang teks
       pages    n_char  n_words
count  240.0     240.0    240.0
mean    20.0   85271.0  12331.0
std     10.0   27514.0   4007.0
min      3.0   10790.0   1546.0
25%     14.0   71119.0  10028.0
50%     18.0   84816.0  12132.0
75%     24.0  102119.0  14733.0
max     69.0  201618.0  28361.0 

   Teks lebih panjang dari batas sel Excel (32.767): 235 dari 240 

3) PDF dengan < 500 karakter per halaman (kemungkinan hasil scan):
   tidak ada 

4) Total kata rusak ligatur: 3
   File dengan ≥ 3 kata rusak: 0


## 8. Menyimpan dataset

In [9]:
kolom = ["articlename", "keyword", "pages", "n_char", "n_words", "textdata"]
urutan_kk = {"Digital servitization": 0, "Ethical leadership": 1, "Dynamic capabilities": 2}
dataset = (df[kolom]
           .assign(_urut=df["keyword"].map(urutan_kk))
           .sort_values(["_urut", "articlename"])
           .drop(columns="_urut")
           .reset_index(drop=True))

# --- CSV (file utama) ---
path_csv = FOLDER_OUTPUT / "journal_fulltext.csv"
dataset.to_csv(path_csv, index=False, encoding="utf-8-sig")

# --- Pickle ---
path_pkl = FOLDER_OUTPUT / "journal_fulltext.pkl"
dataset.to_pickle(path_pkl)

# --- Excel: teks dipecah per 32.000 karakter ---
PANJANG_POTONGAN = 32_000
n_potongan = int(-(-dataset["n_char"].max() // PANJANG_POTONGAN))   # pembulatan ke atas
excel = dataset.drop(columns="textdata").copy()
for k in range(n_potongan):
    excel[f"text_part{k + 1}"] = dataset["textdata"].str.slice(k * PANJANG_POTONGAN, (k + 1) * PANJANG_POTONGAN)

path_xlsx = FOLDER_OUTPUT / "journal_fulltext.xlsx"
with pd.ExcelWriter(path_xlsx, engine="openpyxl") as writer:
    excel.to_excel(writer, sheet_name="dataset", index=False)

for p in (path_csv, path_xlsx, path_pkl):
    print(f"Tersimpan: {p}  ({p.stat().st_size / 1e6:,.1f} MB)")
print("Jumlah kolom potongan teks di Excel:", n_potongan)

Tersimpan: C:\Users\marfu\Documents\LDA_SBMI\journal_fulltext.csv  (20.6 MB)
Tersimpan: C:\Users\marfu\Documents\LDA_SBMI\journal_fulltext.xlsx  (6.3 MB)
Tersimpan: C:\Users\marfu\Documents\LDA_SBMI\journal_fulltext.pkl  (20.6 MB)
Jumlah kolom potongan teks di Excel: 7


## 9. Verifikasi file hasil

Baca ulang ketiga file, lalu pastikan teksnya **identik** dengan DataFrame asli:
- CSV dan Pickle dibandingkan langsung.
- Excel: potongan `text_part1 … text_partN` digabung kembali dulu, lalu dibandingkan.

In [10]:
cek_csv = pd.read_csv(path_csv)
cek_pkl = pd.read_pickle(path_pkl)
cek_xls = pd.read_excel(path_xlsx)
cek_xls["textdata"] = cek_xls.filter(like="text_part").fillna("").astype(str).agg("".join, axis=1)

asli = dataset.set_index("articlename")["textdata"]
for nama, d in [("CSV", cek_csv), ("Pickle", cek_pkl), ("Excel", cek_xls)]:
    sama = d.set_index("articlename")["textdata"].reindex(asli.index).eq(asli).all()
    print(f"{nama:<7} → {len(d)} baris, teks identik: {sama}")

print("\nTeks terpanjang di CSV:", f"{cek_csv['textdata'].str.len().max():,}", "karakter (tidak terpotong)")

CSV     → 240 baris, teks identik: True
Pickle  → 240 baris, teks identik: True
Excel   → 240 baris, teks identik: True

Teks terpanjang di CSV: 201,618 karakter (tidak terpotong)
